In [0]:
from pyspark.sql import functions as F

##### 1.1 预先铺设数据管道


In [0]:
VOLUME_BASE = "/Volumes/workspace/default/olist_files"

##### 1.2 导入订单表


In [0]:
orders_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_orders_dataset.csv")

display(orders_df.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


##### 1.3 导入用户主表


In [0]:
customers_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_customers_dataset.csv")

display(customers_df.limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


##### 1.4 导入商品表


In [0]:
products_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_products_dataset.csv")

display(products_df.limit(5))

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


##### 1.4 导入订单详情表


In [0]:
items_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_order_items_dataset.csv")

display(items_df.limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14


###### 2. 矩阵式注册临时视图（构建内存虚拟数仓）

In [0]:
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")
items_df.createOrReplaceTempView("items")
print("✅ 【数据准备就绪】核心业务表已成功映射为 SQL 临时视图！")

✅ 【数据准备就绪】核心业务表已成功映射为 SQL 临时视图！


##### 3. 编写重工业 SQL 复现 BQ 核心商业指标

In [0]:
BQ_metrics_df = spark.sql("""
    SELECT
        round(SUM(oi.price),2) AS GMV,
        count(DISTINCT(o.order_id)) AS total_orders,
        count(DISTINCT(c.customer_id)) AS total_customers,

        -- 核心效率：客单价 (AOV)
        ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
    FROM  items oi
    INNER JOIN orders o ON o.order_id = oi.order_id
    INNER JOIN customers c ON c.customer_id = o.customer_id
    INNER JOIN products p ON p.product_id = oi.product_id                      
""")

print("\n🚀 【真实指标计算完成】还原真实 Olist 级联后的商业指标：")
display(BQ_metrics_df)


🚀 【真实指标计算完成】还原真实 Olist 级联后的商业指标：


GMV,total_orders,total_customers,avg_order_value
1.35916437E7,98666,98666,137.75
